<a href="https://colab.research.google.com/github/fdhliakbar/IR-Lab/blob/main/P06_Query_Expansion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum ke 6 - Query Expansion



**Query expansion** dalam *Information Retrieval* (IR) adalah teknik yang digunakan untuk meningkatkan
hasil pencarian dengan memperluas kueri pengguna, biasanya dengan menambahkan istilah-istilah
yang relevan atau sinonim dari kata-kata yang ada dalam kueri asli. Tujuannya adalah untuk mengatasi
masalah ketidakcocokan antara istilah yang digunakan oleh pengguna dan istilah yang digunakan
dalam dokumen yang relevan, serta untuk meningkatkan cakupan dan kualitas hasil pencarian.

**Tujuan Query Expansion** :
1. Pengguna mungkin menggunakan istilah yang berbeda dengan istilah yang digunakan dalam
dokumen relevan. Misalnya, pengguna mencari "mobil cepat", sementara dokumen
menggunakan istilah "mobil cepat" atau "mobil kilat".
2. Dengan menambahkan istilah yang lebih umum atau lebih spesifik, query expansion
membantu memperluas hasil pencarian untuk mencakup lebih banyak dokumen relevan.
Sistem Temu Balik Informasi - Informatika – UAD - 2023
53
3. Mengurangi ketidaksesuaian antara kata-kata yang digunakan oleh pengguna dan kata-kata
yang sebenarnya digunakan dalam dokumen yang relevan.

**Jenis-Jenis Query Expansion:**
1. *Manual Query Expansion*: Pengguna secara langsung menambahkan istilah tambahan ke kueri
mereka. Misalnya, setelah melakukan pencarian awal, pengguna mungkin menambahkan
kata-kata tambahan untuk mempersempit hasil.
2. *Automated Query Expansion* (AQE): Sistem pencarian secara otomatis menambahkan istilah terkait atau sinonim berdasarkan berbagai teknik, seperti:
- Sinonim dari kamus leksikal seperti WordNet.
- Relevance Feedback: Sistem menambahkan istilah dari dokumen-dokumen yang dianggap relevan dalam hasil pencarian awal.
- Thesaurus: Menggunakan istilah terkait dari kamus sinonim atau ontologi yang sudah ada.
- Statistical Expansion: Menambahkan istilah yang sering muncul bersamaan dengan kata-kata dalam kueri pengguna dalam dokumen relevan.

**Teknik Query Expansion:**
1. *Relevance Feedback*: Setelah pencarian pertama dilakukan, pengguna dapat memberikan
umpan balik (misalnya, menandai dokumen mana yang relevan). Berdasarkan dokumen yang
dipilih, sistem akan mengekstrak kata-kata tambahan yang sering muncul dalam dokumen
relevan untuk memperluas kueri.
2. *Pseudo Relevance Feedback*: Alih-alih meminta pengguna memberikan umpan balik, sistem
secara otomatis mengasumsikan bahwa beberapa dokumen teratas dari hasil pencarian
pertama adalah relevan, kemudian menggunakan kata-kata dari dokumen tersebut untuk
memperluas kueri.
3. *Semantic Query Expansion*: Sistem menambahkan istilah yang bermakna serupa dengan
menggunakan sumber-sumber semantik seperti WordNet, ontologi, atau kamus sinonim.

In [11]:
# import library standard
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [12]:
!pip install scikit-learn

In [13]:
# import library sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB

# import library sklearn (Evaluasi tak berperingkat)
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# import library sklearn (Evaluasi berperingkat)
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

In [14]:
# import library untuk stemming
!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [15]:
# read dataset
data = pd.read_excel('/content/dataKumparan1.xlsx')
data.head()

,Topic,Title,Content
0,Politik,"Pelanggaran Pemilu, Tiga Caleg di Sulteng Dipr...","Komisioner Bawaslu Sigi, Sulawesi Tengah, Agus..."
1,Politik,"Pemilu Susulan di Kota Jayapura, Suara Jokowi ...",Walaupun dua dari lima distrik melakukan pemil...
2,Politik,"Tsamara Amany Dipinang, Pengurus PSI Daerah Me...","Tsamara Amany, politisi Partai Solidaritas Ind..."
3,Politik,Ada 47 TPS di Sulawesi Utara Berpotensi Pemili...,Badan Pengawas Pemilu (Bawaslu) Provinsi Sulaw...
4,Politik,Ketua KPPS di Sleman Ditemukan Tewas Gantung D...,"Tugiman, Ketua Kelompok Penyelenggara Pemungut..."


In [16]:
# ukuran dataset
print('Ukuran dataset: ', data.shape)

Ukuran dataset:  (60, 3)


In [17]:
# pembagian data training & testing
x_train, x_test, y_train, y_test = train_test_split(data['Content'], data['Topic'], train_size = 0.5, test_size = 0.16)

In [18]:
train_data = pd.DataFrame({'Content': x_train, 'Topic': y_train})
test_data = pd.DataFrame({'Content': x_test, 'Topic': y_test})

In [19]:
df1 = pd.DataFrame(train_data)
print(df1)

                                              Content      Topic
39  Perusahaan e-commerce marketplace Tokopedia ke...  Teknologi
53  Untuk mendukung pengembangan pariwisata Sulawe...     Travel
48  Pariwisata Bintan terus menunjukan perkembanga...     Travel
49  Bila biasanya mesin fotokopi terdapat di kios-...     Travel
28  Masyarakat Indonesia menggunakan hak pilihnya ...  Teknologi
22  Amazon.com mengatakan akan menutup toko daring...  Teknologi
40  Wahana hiburan Trampolin hadir pertama kali di...     Travel
25  Apple ternyata tidak main-main untuk terjun ke...  Teknologi
0   Komisioner Bawaslu Sigi, Sulawesi Tengah, Agus...    Politik
18  Isu kecurangan di Pemilu 2019 terus menyeruak....    Politik
47  Pemerintah Kota Banjarmasin baru saja menyeles...     Travel
58  Setuju atau tidak, ruang bagasi penyimpanan da...     Travel
16  Pemungutan suara Pemilu 2019 telah usai. Tapi ...    Politik
1   Walaupun dua dari lima distrik melakukan pemil...    Politik
50  Untuk pertama kalinya

In [20]:
df2 = pd.DataFrame(test_data)
print(df2)

                                              Content      Topic
15  Mantan Ketua Mahkamah Konstitusi (MK), Mahfud ...    Politik
24  Stasiun Kereta Api Stockholm, Swedia, merupaka...  Teknologi
52  Hari Bumi yang diperingati pada tanggal 22 Apr...     Travel
31  Jelang bulan Ramadhan yang tinggal menghitung ...  Teknologi
7   Badan Pengawas Pemilu (Bawaslu) Kota Banjarmas...    Politik
3   Badan Pengawas Pemilu (Bawaslu) Provinsi Sulaw...    Politik
38  Kamu mungkin pernah merasa kesulitan untuk ber...  Teknologi
27  Pesta demokrasi terbesar di Indonesia resmi di...  Teknologi
19  Cawapres 02 Sandiaga Uno menanggapi rencana ca...    Politik
57  Kabar bahagia datang bagi para penyelam di sel...     Travel


In [21]:
print('ukuran data train: ', train_data.shape)
print('ukuran data test: ', test_data.shape)
n_train = train_data.shape[0]
n_test = test_data.shape[0]

ukuran data train:  (30, 2)
ukuran data test:  (10, 2)


In [22]:
sparse_data = pd.concat([train_data, test_data], ignore_index=True)
df3 = pd.DataFrame(sparse_data)
print(df3)

                                              Content      Topic
0   Perusahaan e-commerce marketplace Tokopedia ke...  Teknologi
1   Untuk mendukung pengembangan pariwisata Sulawe...     Travel
2   Pariwisata Bintan terus menunjukan perkembanga...     Travel
3   Bila biasanya mesin fotokopi terdapat di kios-...     Travel
4   Masyarakat Indonesia menggunakan hak pilihnya ...  Teknologi
5   Amazon.com mengatakan akan menutup toko daring...  Teknologi
6   Wahana hiburan Trampolin hadir pertama kali di...     Travel
7   Apple ternyata tidak main-main untuk terjun ke...  Teknologi
8   Komisioner Bawaslu Sigi, Sulawesi Tengah, Agus...    Politik
9   Isu kecurangan di Pemilu 2019 terus menyeruak....    Politik
10  Pemerintah Kota Banjarmasin baru saja menyeles...     Travel
11  Setuju atau tidak, ruang bagasi penyimpanan da...     Travel
12  Pemungutan suara Pemilu 2019 telah usai. Tapi ...    Politik
13  Walaupun dua dari lima distrik melakukan pemil...    Politik
14  Untuk pertama kalinya

In [23]:
# ukuran sparse data
print('ukuran data test: ', sparse_data.shape)
n_document = sparse_data.shape[0]

ukuran data test:  (40, 2)


# **Preprocessing dengan Stemming dan Stopword**

In [24]:
# create stemmer
StemmerFactory = StemmerFactory()
stemmer = StemmerFactory.create_stemmer()

In [25]:
# stem process
for row in range(n_document):
  sparse_data.loc[row, 'Content'] = stemmer.stem(sparse_data.loc[row, 'Content'])

In [26]:
df4 = pd.DataFrame(sparse_data)
print(df4)

                                              Content      Topic
0   usaha e-commerce marketplace tokopedia kembali...  Teknologi
1   untuk dukung kembang pariwisata sulawesi utara...     Travel
2   pariwisata bintan terus tunjuk kembang yang po...     Travel
3   bila biasa mesin fotokopi dapat di kios yang t...     Travel
4   masyarakat indonesia guna hak pilih dalam milu...  Teknologi
5   amazon com kata akan tutup toko daringnya di c...  Teknologi
6   wahana hibur trampolin hadir pertama kali di k...     Travel
7   apple nyata tidak main untuk terjun ke industr...  Teknologi
8   komisioner bawaslu sigi sulawesi tengah agus s...    Politik
9   isu curang di milu 2019 terus seruak dua timse...    Politik
10  perintah kota banjarmasin baru saja selesai de...     Travel
11  tuju atau tidak ruang bagasi simpan dalam kabi...     Travel
12  mungut suara milu 2019 telah usai tapi duka ma...    Politik
13  walaupun dua dari lima distrik laku pilih umum...    Politik
14  untuk pertama kali da

# **Perhitungan Bobot**

In [27]:
vectorizer = CountVectorizer()
tf = vectorizer.fit_transform(sparse_data['Content'])
print(['Jumlah dokumen: ', tf.shape[0]])
print(['Jumlah term: ', tf.shape[1]])

['Jumlah dokumen: ', 40]
['Jumlah term: ', 2338]


In [28]:
print('Daftar Term:')
vectorizer.get_feature_names_out()

Daftar Term:


array(['00', '000', '0004', ..., 'zat', 'ziarah', 'zoetry'], dtype=object)

In [29]:
print('Daftar Stopword:')
vectorizer.get_stop_words()

Daftar Stopword:


In [30]:
print('Matriks Tf:')
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names_out())
tf_matrix

Matriks Tf:


,00,000,0004,01,02,039,04,043,052,056,...,yesus,yogyakarta,yunani,yusuf,zahid,zainuddin,zamih,zat,ziarah,zoetry
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,2,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [31]:
print('Matriks Tf (khusus data train):')
tf_train = tf_matrix[:n_train]
tf_train.shape

Matriks Tf (khusus data train):


(30, 2338)

In [32]:
transformer = TfidfTransformer(sublinear_tf=True)

# Penyesuaian df agar query (data test) tidak dihitung pada perhitungan df
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print('Jumlah dokumen:', weight.shape[0])
print('Jumlah term:', weight.shape[1])

Jumlah dokumen: 40
Jumlah term: 2338


In [33]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names_out())
weight_matrix

,00,000,0004,01,02,039,04,043,052,056,...,yesus,yogyakarta,yunani,yusuf,zahid,zainuddin,zamih,zat,ziarah,zoetry
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
6,0.099235,0.062267,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000


In [34]:
# pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

# **Perhitungan Cosine Similarity**

In [35]:
# perhitungan Cosine Similarity
cosim = cosine_similarity(weight_train, weight_test)
print('Ukuran matriks cosine similarity:', cosim.shape)

Ukuran matriks cosine similarity: (30, 10)


In [36]:
name = []
for i in range(n_test):
  name.append('Dokumen ' + str(i))

In [37]:
# baris = dokumen train, kolom = dokumen test
print('Matriks Cosine Similarity:')
cosim_matrix = pd.DataFrame(cosim, columns=name)
cosim_matrix.shape

Matriks Cosine Similarity:


(30, 10)

In [38]:
train_data

,Content,Topic
39,Perusahaan e-commerce marketplace Tokopedia ke...,Teknologi
53,Untuk mendukung pengembangan pariwisata Sulawe...,Travel
48,Pariwisata Bintan terus menunjukan perkembanga...,Travel
49,Bila biasanya mesin fotokopi terdapat di kios-...,Travel
28,Masyarakat Indonesia menggunakan hak pilihnya ...,Teknologi
22,Amazon.com mengatakan akan menutup toko daring...,Teknologi
40,Wahana hiburan Trampolin hadir pertama kali di...,Travel
25,Apple ternyata tidak main-main untuk terjun ke...,Teknologi
0,"Komisioner Bawaslu Sigi, Sulawesi Tengah, Agus...",Politik
18,Isu kecurangan di Pemilu 2019 terus menyeruak....,Politik


In [39]:
# cosim matrix + label
cosim_matrix['Label Train'] = train_data['Topic'].values
label_row = dict(zip(name, test_data['Topic'].values))
label_cosim = pd.concat([cosim_matrix, pd.DataFrame([label_row], index=['Label Test'])], ignore_index=False)
label_cosim.rename({label_cosim.index[-1]:'Label Test'}, inplace=True)


label_test = pd.DataFrame(label_cosim.iloc[-1])
label_test = label_test.T
label_test

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
Label Test,Politik,Teknologi,Travel,Teknologi,Politik,Politik,Teknologi,Teknologi,Politik,Travel,NaN


In [40]:
label_cosim

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
0,0.068152,0.088459,0.064976,0.086198,0.088985,0.074873,0.121725,0.078144,0.073514,0.099127,Teknologi
1,0.104586,0.088944,0.102238,0.096698,0.089221,0.154901,0.103907,0.07603,0.081391,0.125866,Travel
2,0.086769,0.099876,0.08614,0.094247,0.062046,0.051301,0.094879,0.068871,0.073617,0.128154,Travel
3,0.079579,0.09081,0.07422,0.069277,0.063969,0.069369,0.105848,0.066152,0.075472,0.0853,Travel
4,0.22381,0.090512,0.064078,0.107329,0.165828,0.094585,0.120586,0.173931,0.158727,0.143621,Teknologi
5,0.085963,0.062669,0.068838,0.086937,0.076081,0.02892,0.134322,0.059026,0.071777,0.102501,Teknologi
6,0.091608,0.093769,0.084232,0.085375,0.093701,0.05094,0.092602,0.049656,0.091144,0.08435,Travel
7,0.078069,0.096742,0.091676,0.134351,0.056069,0.038489,0.111721,0.065281,0.068111,0.093842,Teknologi
8,0.119412,0.046494,0.060198,0.056658,0.149199,0.113815,0.100344,0.068932,0.080859,0.080097,Politik
9,0.161587,0.076583,0.049011,0.087431,0.112317,0.050116,0.095689,0.064774,0.139818,0.071373,Politik


In [41]:
cosim_matrix

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
0,0.068152,0.088459,0.064976,0.086198,0.088985,0.074873,0.121725,0.078144,0.073514,0.099127,Teknologi
1,0.104586,0.088944,0.102238,0.096698,0.089221,0.154901,0.103907,0.076030,0.081391,0.125866,Travel
2,0.086769,0.099876,0.086140,0.094247,0.062046,0.051301,0.094879,0.068871,0.073617,0.128154,Travel
3,0.079579,0.090810,0.074220,0.069277,0.063969,0.069369,0.105848,0.066152,0.075472,0.085300,Travel
4,0.223810,0.090512,0.064078,0.107329,0.165828,0.094585,0.120586,0.173931,0.158727,0.143621,Teknologi
5,0.085963,0.062669,0.068838,0.086937,0.076081,0.028920,0.134322,0.059026,0.071777,0.102501,Teknologi
6,0.091608,0.093769,0.084232,0.085375,0.093701,0.050940,0.092602,0.049656,0.091144,0.084350,Travel
7,0.078069,0.096742,0.091676,0.134351,0.056069,0.038489,0.111721,0.065281,0.068111,0.093842,Teknologi
8,0.119412,0.046494,0.060198,0.056658,0.149199,0.113815,0.100344,0.068932,0.080859,0.080097,Politik
9,0.161587,0.076583,0.049011,0.087431,0.112317,0.050116,0.095689,0.064774,0.139818,0.071373,Politik


In [42]:
# prompt: buatkan tampilan cosim_matrix secara descending

# Sort the cosim_matrix by all columns in descending order
cosim_matrix_sorted = cosim_matrix.sort_values(by=cosim_matrix.columns.tolist(), ascending=False)
cosim_matrix_sorted

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
28,0.250578,0.107833,0.093020,0.119583,0.186025,0.155364,0.151439,0.126700,0.250126,0.104272,Politik
4,0.223810,0.090512,0.064078,0.107329,0.165828,0.094585,0.120586,0.173931,0.158727,0.143621,Teknologi
15,0.217953,0.072698,0.068561,0.074305,0.138052,0.067930,0.072549,0.123163,0.173687,0.090725,Politik
12,0.162855,0.055457,0.051467,0.083306,0.133089,0.130100,0.083600,0.090160,0.105142,0.090826,Politik
9,0.161587,0.076583,0.049011,0.087431,0.112317,0.050116,0.095689,0.064774,0.139818,0.071373,Politik
17,0.155806,0.104229,0.106274,0.118675,0.107072,0.083289,0.166136,0.088589,0.127465,0.126952,Politik
13,0.129897,0.075231,0.062844,0.086375,0.117647,0.143210,0.088296,0.086452,0.177308,0.071964,Politik
22,0.127544,0.085530,0.144546,0.098190,0.069376,0.057228,0.125060,0.077671,0.118580,0.103385,Travel
29,0.123683,0.070103,0.060572,0.071784,0.134407,0.069470,0.096494,0.081867,0.138465,0.082919,Politik
27,0.119585,0.116162,0.103086,0.137726,0.089968,0.051411,0.177909,0.118049,0.093037,0.125358,Teknologi


In [44]:
import os

directory = '/content/drive/MyDrive/STBI'
if not os.path.exists(directory):
    os.makedirs(directory)

cosim_matrix.to_csv(os.path.join(directory, 'cosim_matrix.csv'), index=False)
cosim_matrix_sorted.to_csv(os.path.join(directory, 'cosim_matrix_sorted.csv'), index=False)

In [45]:
def average_precision(cosim_matrix, n_retrieve):
  average_precision = []

  # loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    # sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
    # print(top_n)

    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
    # print(relevant)

    # list of precision
    precision = []
    peringkat = 0
    counter_relevant = 0

    for r in relevant:
      peringkat +=1
      if r == True:
        counter_relevant += 1
        precision.append(counter_relevant/peringkat)

    average_precision.append(np.mean(precision))

    # print('Average Precision: ', np.mean(precision))
    # print(average_precision)

  return average_precision

def mean_average_precision(cosim_matrix, n_retrieve):
  ap = average_precision(cosim_matrix, n_retrieve)
  map = np.mean(ap)
  return map

def precision_at_k(cosim_matrix, k_retrieve):
  precision_at_k = []

  # loop utk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    # sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:k_retrieve]
    # print(top_n)

    # list of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
    # print(relevant)

    # list of precision
    precision = np.sum(relevant)/len(relevant)
    precision_at_k.append(precision)
    # print(precision_at_k)

  return precision_at_k

def r_precision(cosim_matrix, n_retrieve):
    r_precision = []

    # loop utk mengambil setiap query di cosim matrix
    for column in cosim_matrix.iloc[:, :-1]:
      # sort and get top n
      sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
      top_n = sorted_cosim.iloc[:n_retrieve]
      # print(top_n)

      # list of relevance
      relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
      # print(relevant

      # list of precision
      precision = np.sum(relevant)/len(relevant)
      r_precision.append(precision)
    return r_precision

In [46]:
print('\nHasil Evaluasi MAP: ', mean_average_precision(cosim_matrix, 10))
print('Hasil Evaluasi Precision@k: ', precision_at_k(cosim_matrix, 10))
print('Hasil Evaluasi R-Precision: ', r_precision(cosim_matrix, 10))


Hasil Evaluasi MAP:  0.7846291572184431
Hasil Evaluasi Precision@k:  [np.float64(0.7), np.float64(0.4), np.float64(0.5), np.float64(0.4), np.float64(0.8), np.float64(0.6), np.float64(0.6), np.float64(0.5), np.float64(0.6), np.float64(0.5)]
Hasil Evaluasi R-Precision:  [np.float64(0.7), np.float64(0.4), np.float64(0.5), np.float64(0.4), np.float64(0.8), np.float64(0.6), np.float64(0.6), np.float64(0.5), np.float64(0.6), np.float64(0.5)]


In [47]:
numbers = [
    0.164109, 0.054123, 0.138920,
    0.084240, 0.084468, 0.066134,
    0.063556
]

mean_value = sum(numbers) / 10
mean_value


0.065555

## POSTEST 6 - Update Program & Evaluasi

1. Analisa hasil `query` yang diperluas, jelaskan mengapa hasil
`recall` meningkat namun hasil `precision` menurun?

Note: Jika ada perlu ditanyakan terkait teknis praktikum, jangan ragu bertanya. Silahkan bertanya di group atau pc dengan asisten `Fadhli` & `Aufa`

### Selamat Mengerjakan 😺